# 06 Predict

다음 회차의 후보 보상 상장 여부를 과거 패턴만으로 예측하는 누수 없는 실험 노트북이다. maplessunday의 블랙박스 예측과 달리 여기서는 재현 가능한 후보 행렬, 시간순 분할, 베이스라인, 모델 2종, 다음 회차 추론 결과를 한 흐름에서 비교한다.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.predict_dataset import build_inference_frame, write_prediction_frame

RANDOM_SEED = 42
SPLIT_CUTOFF = pd.Timestamp("2025-01-01")


## 데이터 로드

`data/processed/tidy_rewards.csv`와 `data/interim/predict_dataset.csv`만 사용한다. `data/raw`는 읽거나 수정하지 않는다.

후보 상위 N개는 train 구간(`< 2025-01-01`) 빈도로만 선정한다. 테스트 구간 정보가 후보 universe 설계에 들어가지 않도록 같은 컷오프를 후보 선정과 시간순 분할에 함께 사용한다.

In [ ]:
predict_path = ROOT / "data" / "interim" / "predict_dataset.csv"
tidy_path = ROOT / "data" / "processed" / "tidy_rewards.csv"

write_prediction_frame(
    tidy_path=tidy_path,
    output_path=predict_path,
    candidate_cutoff=SPLIT_CUTOFF,
)

df = pd.read_csv(predict_path, parse_dates=["date"])
summary = {
    "행": len(df),
    "회차": df["date"].nunique(),
    "후보": df["reward_norm"].nunique(),
    "양성비율": df["y"].mean(),
    "시작일": df["date"].min().date(),
    "종료일": df["date"].max().date(),
    "후보컷오프": SPLIT_CUTOFF.date(),
}
summary

## 시간순 분할

랜덤 분할은 사용하지 않는다. 기본 컷오프는 `2025-01-01`이며, 학습 구간의 마지막 날짜가 테스트 구간의 첫 날짜보다 앞서는지 확인한다.

In [ ]:
cutoff = SPLIT_CUTOFF
train = df.loc[df["date"] < cutoff].copy()
test = df.loc[df["date"] >= cutoff].copy()

split_summary = {
    "학습행": len(train),
    "테스트행": len(test),
    "학습시작일": train["date"].min().date(),
    "학습종료일": train["date"].max().date(),
    "테스트시작일": test["date"].min().date(),
    "테스트종료일": test["date"].max().date(),
    "학습양성비율": train["y"].mean(),
    "테스트양성비율": test["y"].mean(),
    "분할검증": train["date"].max() < test["date"].min(),
}
assert train["date"].max() < test["date"].min()
split_summary

## 평가 함수

점수는 확률 또는 확률에 준하는 연속값을 받는다. 회차별 적중률은 각 회차의 실제 양성 개수만큼 상위 후보를 골랐을 때 하나 이상 실제 양성을 포함한 회차 비율이다.

점수가 동점이면 `nlargest`가 상위 k개를 고르며, 이 값은 베이스라인 비교를 위한 근사 지표로만 해석한다.

In [ ]:
def evaluate(y_true, y_score, dates):
    y_true = pd.Series(y_true).astype(int).reset_index(drop=True)
    y_score = pd.Series(y_score).astype(float).reset_index(drop=True)
    dates = pd.Series(dates).reset_index(drop=True)
    y_pred = (y_score >= 0.5).astype(int)
    work = pd.DataFrame({"date": dates, "y": y_true, "score": y_score})

    def date_hit(group):
        k = int(group["y"].sum())
        if k < 1:
            return np.nan
        return float(group.nlargest(k, "score")["y"].max())

    auc = roc_auc_score(y_true, y_score) if y_true.nunique() > 1 else np.nan
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": auc,
        "date_hit_rate": work.groupby("date", sort=True).apply(date_hit).mean(),
    }

## 베이스라인

B1은 `days_since_last >= roll_gap_median` 규칙이다. B2는 과거 출현 비율 `past_rate`를 사전확률 점수로 사용한다. 이벤트 지표, 중단 여부, 미래 파생 변수는 사용하지 않는다.

`category`는 보상의 정적 속성으로 보고 전체 이력의 최빈값을 사용하며, 예측 시점에 알려진 값으로 간주한다.

In [ ]:
b1_score = (
    test["roll_gap_median"].notna()
    & (test["days_since_last"] >= test["roll_gap_median"])
).astype(float)
b2_score = test["past_rate"].clip(0, 1)

baseline_results = pd.DataFrame(
    [
        {"기준선": "간격규칙", **evaluate(test["y"], b1_score, test["date"])},
        {"기준선": "사전확률", **evaluate(test["y"], b2_score, test["date"])},
    ]
)
baseline_results

## 모델 입력 준비

`X_train`, `y_train`, `X_test`, `y_test`가 준비되어 있다. 로지스틱 회귀나 트리 계열 모델을 직접 작성하고 `predict_proba` 또는 이에 준하는 점수를 `evaluate()`로 베이스라인과 비교한다.

In [ ]:
feature_columns = [
    "days_since_last",
    "past_count",
    "past_rate",
    "roll_gap_mean",
    "roll_gap_median",
    "lag1_appeared",
    "category",
    "month",
    "quarter",
    "weekofyear",
]

X_all = pd.get_dummies(df[feature_columns], columns=["category"], dummy_na=True)
X_all = X_all.fillna(-1)
X_train = X_all.loc[train.index]
X_test = X_all.loc[test.index]
y_train = train["y"].astype(int)
y_test = test["y"].astype(int)

X_train.shape, X_test.shape, y_train.mean(), y_test.mean()

## 모델 학습과 시간순 백테스트

모델은 로지스틱 회귀와 랜덤포레스트 2종만 사용한다. 두 모델 모두 `random_state=42`로 고정하고, 학습은 `2025-01-01` 이전 행만 사용한다. 평가는 베이스라인과 같은 `evaluate()` 함수로 테스트 구간 `2025-01-05`부터 `2026-06-07`까지 비교한다.


In [ ]:
logistic_model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_SEED)
tree_model = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=RANDOM_SEED)

logistic_model.fit(X_train, y_train)
tree_model.fit(X_train, y_train)

logistic_score = logistic_model.predict_proba(X_test)[:, 1]
tree_score = tree_model.predict_proba(X_test)[:, 1]

model_results = pd.DataFrame(
    [
        {"방법": "로지스틱", **evaluate(y_test, logistic_score, test["date"])},
        {"방법": "랜덤포레스트", **evaluate(y_test, tree_score, test["date"])},
    ]
)
comparison_results = pd.concat(
    [baseline_results.rename(columns={"기준선": "방법"}), model_results],
    ignore_index=True,
)
comparison_results


## 다음 회차 추론

관측 가능한 마지막 회차는 `2026-06-07`이고, 다음 미관측 회차는 `2026-06-14`이다. 같은 후보 universe와 같은 피처 정의를 사용하며, 원핫 인코딩 후 학습 입력 컬럼 순서로 다시 맞춘다.


In [ ]:
TARGET_DATE = pd.Timestamp("2026-06-14")
infer = build_inference_frame(pd.read_csv(tidy_path), TARGET_DATE, candidate_cutoff=SPLIT_CUTOFF)
X_infer = pd.get_dummies(infer[feature_columns], columns=["category"], dummy_na=True)
X_infer = X_infer.reindex(columns=X_all.columns, fill_value=0)
X_infer = X_infer.fillna(-1)

assert list(X_infer.columns) == list(X_all.columns)

infer_score = tree_model.predict_proba(X_infer)[:, 1]
prediction_table = infer[["date", "reward_norm"]].copy()
prediction_table["상장확률"] = infer_score
prediction_table = prediction_table.sort_values("상장확률", ascending=False).reset_index(drop=True)
prediction_table["순위"] = np.arange(1, len(prediction_table) + 1)
prediction_table["구분"] = np.where(prediction_table["순위"] <= 5, "고신뢰", "불확실")
prediction_table = prediction_table[["순위", "date", "reward_norm", "상장확률", "구분"]]
prediction_table


상위 5개는 이 모델 안에서 상대적으로 점수가 높은 고신뢰 후보로 분리하고, 나머지는 불확실 후보로 둔다. 이 기준은 운영상 검토를 쉽게 하기 위한 절단선이며 실제 상장 보장을 뜻하지 않는다. 확률값은 후보 20개 내부의 상대 점수로 읽어야 하며, 이벤트성 보상은 과거 주기 신호만으로 설명하기 어렵다.


## 결론 프레임

이 노트북의 예측은 다음 회차 후보 20개를 과거 출현 패턴으로 정렬하는 보조 지표다. 예측 가능한 부분은 거의 매주 반복되는 상시 보상 패턴이고, 예측하기 어려운 부분은 스페셜 이벤트나 운영상 편성 판단처럼 과거 보상 주기만으로 설명되지 않는 영역이다.

05_model의 주기 기준 실험에서는 보상별 중앙값 기준 MAE가 74.49로 전체 중앙값 기준 93.65보다 낮았다. 이는 주기가 일부 신호를 갖는다는 근거지만, 특정 보상을 확정 선택할 정도의 근거는 아니다. 따라서 아래 결과는 확정 예보가 아니라 재현 가능한 후보 정렬로 해석한다.

maplessunday와의 차이는 성능 우위 주장이 아니라 투명성과 재현성이다. 같은 입력 데이터, 같은 시간순 분할, 같은 후보 universe에서 산출 과정을 다시 실행할 수 있다는 점을 프로젝트의 차별점으로 둔다.
